<a href="https://colab.research.google.com/github/makaog5-sudo/CRM-SP500/blob/main/CRM_stock_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install yfinance pandas numpy


In [14]:
import pandas as pd, numpy as np, yfinance as yf, sklearn, scipy
from datetime import datetime, timezone


print("pandas", pd.__version__, "| numpy", np.__version__, "| yfinance", yf.__version__,
      "| sklearn", sklearn.__version__, "| scipy", scipy.__version__)

# --- Task 1.1: Extract S&P 500 tickers from Wikipedia ---
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

tables = pd.read_html(url, storage_options={"User-Agent": "Mozilla/5.0"})
sp500 = next(
    table for table in tables
    if "Symbol" in table.columns and "Security" in table.columns
)

retrieval_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")

sp500["Yahoo_Ticker"] = sp500["Symbol"].str.replace(".", "-", regex=False)

# Save the list with its retrieval date
sp500["Retrieval_Date_UTC"] = retrieval_date
sp500.to_csv("sp500_tickers.csv", index=False)

tickers = sp500["Yahoo_Ticker"].tolist()

print("Retrieval date:", retrieval_date)
print("Total tickers:", len(tickers))
print("First 10 tickers:", tickers[:10])

sp500[["Symbol", "Yahoo_Ticker", "Security", "GICS Sector"]].head()

pandas 2.2.3 | numpy 2.4.6 | yfinance 0.2.66 | sklearn 1.6.1 | scipy 1.16.3
Retrieval date: 2026-11-21
Total tickers: 503
First 10 tickers: ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']


,Symbol,Yahoo_Ticker,Security,GICS Sector
0,MMM,MMM,3M,Industrials
1,AOS,AOS,A. O. Smith,Industrials
2,ABT,ABT,Abbott Laboratories,Health Care
3,ABBV,ABBV,AbbVie,Health Care
4,ACN,ACN,Accenture,Information Technology


In [11]:

# Download all S&P 500 stocks and the benchmark
all_tickers = tickers + ["^GSPC"]

stock_data = yf.download(
    all_tickers,
    start="2021-01-01",
    end="2025-01-01",
    auto_adjust=False,
    group_by="column",
    threads=True,
    progress=True
)

print("Dataset shape:", stock_data.shape)
print("Number of trading days:", len(stock_data))

stock_data.head()


[*********************100%***********************]  504 of 504 completed
ERROR:yfinance:
5 Failed downloads:
ERROR:yfinance:['VYLR', 'Q', 'FDXF', 'SNDK', 'HONA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Dataset shape: (1005, 3024)
Number of trading days: 1005


Price        Adj Close                                                \
Ticker               A        AAPL       ABBV        ABNB        ABT   
Date                                                                   
2021-01-04  114.009125  125.537117  84.219131  139.149994  97.269714   
2021-01-05  114.941269  127.089233  85.090019  148.300003  98.473198   
2021-01-06  118.093231  122.811203  84.354950  142.770004  98.268166   
2021-01-07  121.235603  127.001915  85.257790  151.270004  99.222054   
2021-01-08  122.100456  128.098129  85.705208  149.770004  99.498405   

Price                                                                 ...  \
Ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                  ...   
2021-01-04  33.186359  232.029343  485.339996  132.622604  42.189491  ...   
2021-01-05  33.319485  233.350281  485.690002  134.993179  43.010105  ...   
2021-01-06  34.783871  235.901642  466.309998  135.601715  44.752842  ...   
2021-01-07  34.460564  238.127304  477.739990  141.332748  44.355217  ...   
2021-01-08  34.650742  238.995834  485.100006  142.359070  44.473656  ...   

Price        Volume                                                          \
Ticker         WYNN      XEL       XOM      XYL       XYZ      YUM      ZBH   
Date                                                                          
2021-01-04  3596800  3197100  27764700   814700   9838400  2058200  1265870   
2021-01-05  1814600  1976800  44035100  1005200   9699400  1475300   998894   
2021-01-06  2433700  2667000  36484800  1015300  12003500  1352700  1424799   
2021-01-07  3846700  2630400  29528100  1539300  10214200  2222900  1237648   
2021-01-08  2812400  2221500  28628200  1632200   9102100  1252200  1259072   

Price                                    
Ticker        ZBRA      ZTS       ^GSPC  
Date                                     
2021-01-04  432600  1805900  5015000000  
2021-01-05  374700  1358600  4591020000  
2021-01-06  419500  2139800  6064110000  
2021-01-07  379900  1710700  5099160000  
2021-01-08  348500  1475900  4773040000  

[5 rows x 3024 columns]

In [12]:
raw = stock_data
stock_data.to_pickle("raw_prices_2021_2024.pkl.gz")

print("Stock data saved successfully!")


Stock data saved successfully!


In [13]:
%cd /content

/content


In [15]:
# --- Task 1.2: Data validation and exclusion log ---
raw = stock_data
adj, close, vol = raw["Adj Close"], raw["Close"], raw["Volume"]
bench_dates = adj["^GSPC"].dropna().index
print("Duplicate dates:", raw.index.duplicated().sum())

exclusions = []
for t in tickers:
    if t not in adj.columns or adj[t].isna().all():
        exclusions.append((t, "No Yahoo data for 2021-2024 (failed download)")); continue
    a, c, v = adj[t].reindex(bench_dates), close[t].reindex(bench_dates), vol[t].reindex(bench_dates)
    if a.first_valid_index() > bench_dates[0]:
        exclusions.append((t, f"Incomplete history: first price {a.first_valid_index().date()}"))
    elif a.isna().any() or c.isna().any() or v.isna().any():
        exclusions.append((t, f"Missing values inside period ({int(a.isna().sum())} days)"))
    elif (a <= 0).any() or (c <= 0).any():
        exclusions.append((t, "Non-positive price"))
    elif (v <= 0).any():
        exclusions.append((t, f"Zero volume on {int((v <= 0).sum())} days"))

excl_log = pd.DataFrame(exclusions, columns=["Ticker", "Reason"])
included = [t for t in tickers if t not in set(excl_log["Ticker"])]
excl_log.to_csv("exclusion_log.csv", index=False)
print(f"Included: {len(included)} | Excluded: {len(excl_log)}")

moves = close[included].pct_change(fill_method=None).abs()
print("Possible corporate-action days (|Close move| > 40%):\n", moves[moves > 0.4].stack())
excl_log

Duplicate dates: 0
Included: 486 | Excluded: 17
Possible corporate-action days (|Close move| > 40%):
 Date        Ticker
2022-08-05  CVNA      0.400716
2022-12-07  CVNA      0.429210
2023-01-12  CVNA      0.460000
2023-06-08  CVNA      0.560206
2023-07-19  CVNA      0.402010
2024-04-11  GL        0.531402
2024-07-26  DXCM      0.406583
2024-11-15  BE        0.591867
dtype: float64


,Ticker,Reason
0,APP,Incomplete history: first price 2021-04-15
1,COIN,Incomplete history: first price 2021-04-14
2,CEG,Incomplete history: first price 2022-01-19
3,EXE,Incomplete history: first price 2021-02-10
4,FDXF,No Yahoo data for 2021-2024 (failed download)
5,FERG,Zero volume on 27 days
6,GEHC,Incomplete history: first price 2022-12-15
7,GEV,Incomplete history: first price 2024-03-27
8,HONA,No Yahoo data for 2021-2024 (failed download)
9,Q,No Yahoo data for 2021-2024 (failed download)
